# Tahap 1 - Membangun Case Base

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = DATA_DIR / "results"
LOGS_DIR = BASE_DIR / "logs"

# Create folders if not exist
for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

BASE_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang
RAW_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/raw
PROCESSED_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed


In [2]:
# Membuat template inventory untuk 40 putusan PN Tangerang - Pidana Umum Pencurian

jumlah_putusan = 40

inventory = pd.DataFrame({
    "case_id": [f"case_{i:03d}" for i in range(1, jumlah_putusan + 1)],
    "no_perkara": ["" for _ in range(jumlah_putusan)],
    "pengadilan": ["PN Tangerang" for _ in range(jumlah_putusan)],
    "jenis_perkara": ["Pidana Umum - Pencurian" for _ in range(jumlah_putusan)],
    "tanggal_putusan": ["" for _ in range(jumlah_putusan)],
    "sumber_url": ["" for _ in range(jumlah_putusan)],
    "raw_file": [f"case_{i:03d}.txt" for i in range(1, jumlah_putusan + 1)],
    "status_download": ["belum" for _ in range(jumlah_putusan)],
    "jumlah_kata": [0 for _ in range(jumlah_putusan)]
})

inventory_path = PROCESSED_DIR / "case_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("Template inventory berhasil dibuat:")
print(inventory_path)

inventory.head()

Template inventory berhasil dibuat:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,,PN Tangerang,Pidana Umum - Pencurian,,,case_001.txt,belum,0
1,case_002,,PN Tangerang,Pidana Umum - Pencurian,,,case_002.txt,belum,0
2,case_003,,PN Tangerang,Pidana Umum - Pencurian,,,case_003.txt,belum,0
3,case_004,,PN Tangerang,Pidana Umum - Pencurian,,,case_004.txt,belum,0
4,case_005,,PN Tangerang,Pidana Umum - Pencurian,,,case_005.txt,belum,0


In [6]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

# Baca CSV sebagai teks semua agar tidak error saat isi nomor perkara
df = pd.read_csv(inventory_path, dtype=str).fillna("")

# Pastikan kolom penting ada
required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

# Semua kolom dibuat string dulu supaya aman
for col in required_columns:
    df[col] = df[col].astype(str)

case_id = "case_006"

# Isi data case_001
df.loc[df["case_id"] == case_id, "no_perkara"] = "310/Pid.B/2019/PN.Tng"
df.loc[df["case_id"] == case_id, "tanggal_putusan"] = "18-03-2019"
df.loc[df["case_id"] == case_id, "pengadilan"] = "PN Tangerang"
df.loc[df["case_id"] == case_id, "jenis_perkara"] = "Pidana Umum - Pencurian"
df.loc[df["case_id"] == case_id, "sumber_url"] = "https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html"
df.loc[df["case_id"] == case_id, "raw_file"] = "case_006.txt"
df.loc[df["case_id"] == case_id, "status_download"] = "belum"
df.loc[df["case_id"] == case_id, "jumlah_kata"] = "0"

# Simpan ulang
df.to_csv(inventory_path, index=False)

print("Data berhasil diperbarui dan disimpan ke:")
print(inventory_path)

df[df["case_id"] == case_id]

Data berhasil diperbarui dan disimpan ke:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
5,case_006,310/Pid.B/2019/PN.Tng,PN Tangerang,Pidana Umum - Pencurian,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0


In [9]:
rows_text = """
586/Pid.B/2019/PN Tng	16 Mei 2019	https://putusan3.mahkamahagung.go.id/direktori/putusan/08310eb6ee84116780cee64575f7054e.html
2203/PID.B/2014/PN. TNG	6 Januari 2015	https://putusan3.mahkamahagung.go.id/direktori/putusan/84cb49872067c701cfdf1ec9ba18e2d7.html
105/Pid.B/2024/PN Tng	27 Februari 2024	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaef333448cd3426a003303334383332.html
465/Pid.B/2012/PN. TNG	12 April 2012	https://putusan3.mahkamahagung.go.id/direktori/putusan/5447530578b1a72d2ab822129b791116.html
1347 / PID.B / 2014 / PN.TNG.	20 Agustus 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/a3aedfdaa5c381abd8efe3088559cfb4.html
672 / PID.B / 2013 / PN.TNG	15 Mei 2013	https://putusan3.mahkamahagung.go.id/direktori/putusan/af8fac0ea6bf31bb3475f4a607d8f0b7.html
395/Pid.B/2023/PN Tng	9 Mei 2023	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedfadf877aa55c8478313633353437.html
46/Pid.B/2024/PN Tng	5 Maret 2024	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaef3327a10bcfc4a212303231373537.html
1431/Pid.B/2021/PN Tng	30 Nopember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec527fe57c45ac863c313532333030.html
1544/Pid.B/2021/PN Tng	17 Nopember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec47891274e3d6889b313633303538.html
1048/Pid.B/2022/PN Tng	19 September 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed38c2290e398c9854313535363438.html
1952/Pid.B/2022/PN Tng	20 Desember 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed803e5dbc637a802e313531343436.html
1649/Pid.B/2022/PN Tng	30 Nopember 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed708d0c8782529978313535373432.html
1905/Pid.B/2021/PN Tng	19 Januari 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec7cbf4158b9c8bc31303934323231.html
1871/Pid.B/2021/PN Tng	5 Januari 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec6ed1abbff3e08ad8313631383535.html
137/Pid.B/2022/PN Tng	13 April 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaecbafe85947050b5c0313435313236.html
893/Pid.B/2022/PN Tng	30 Juni 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaecf918e98122fe867a313533363333.html
235/Pid.B/2017/PN.TNG.	12 April 2017	https://putusan3.mahkamahagung.go.id/direktori/putusan/31651f98aaf5fb0c522ce3ca89d60b7f.html
1686/Pid.B/2022/PN Tng	30 Nopember 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed7b9067f76cd497b3313631393236.html
168/Pid.B/2023/PN Tng	8 Maret 2023	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedbe34bbc79b18b4cf313134323031.html
32/Pdt.P/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4e3a9c5cb8ef323032343033.html
292/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4c54b6fcbc41323032343030.html
272/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4a5d78e89838323032333536.html
"""

In [10]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

for col in required_columns:
    df[col] = df[col].astype(str)

rows = []

for line in rows_text.strip().splitlines():
    parts = line.strip().split("\t")
    
    if len(parts) >= 3:
        no_perkara = parts[0].strip()
        tanggal_putusan = parts[1].strip()
        url = parts[2].strip()
        
        if url.startswith("http") and "/direktori/putusan/" in url:
            rows.append({
                "no_perkara": no_perkara,
                "tanggal_putusan": tanggal_putusan,
                "sumber_url": url
            })

seen = set()
clean_rows = []

for row in rows:
    if row["sumber_url"] not in seen:
        seen.add(row["sumber_url"])
        clean_rows.append(row)

print(f"Jumlah data valid dari browser: {len(clean_rows)}")

existing_urls = set(df["sumber_url"].dropna().astype(str).tolist())

added = 0

for row in clean_rows:
    url = row["sumber_url"]

    if url in existing_urls:
        print(f"Dilewati karena sudah ada: {url}")
        continue

    empty_rows = df.index[(df["sumber_url"] == "") | (df["sumber_url"].isna())].tolist()

    if not empty_rows:
        print("Tidak ada baris kosong lagi.")
        break

    idx = empty_rows[0]
    case_id = df.loc[idx, "case_id"]

    if case_id == "" or case_id.lower() == "nan":
        case_id = f"case_{idx+1:03d}"
        df.loc[idx, "case_id"] = case_id

    df.loc[idx, "no_perkara"] = row["no_perkara"]
    df.loc[idx, "tanggal_putusan"] = row["tanggal_putusan"]
    df.loc[idx, "pengadilan"] = "PN Tangerang"
    df.loc[idx, "jenis_perkara"] = "Pidana Umum - Pencurian"
    df.loc[idx, "sumber_url"] = url
    df.loc[idx, "raw_file"] = f"{case_id}.txt"
    df.loc[idx, "status_download"] = "belum"
    df.loc[idx, "jumlah_kata"] = "0"

    existing_urls.add(url)
    added += 1

df.to_csv(inventory_path, index=False)

print(f"Berhasil menambahkan {added} data baru.")
print("File disimpan ke:", inventory_path)

df[["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file", "status_download", "jumlah_kata"]].head(40)

Jumlah data valid dari browser: 23
Tidak ada baris kosong lagi.
Berhasil menambahkan 17 data baru.
File disimpan ke: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0
